# Практическая работа 7. Задание 3
A* с заданной эвристикой: проверка допустимости, сравнение с h = 0

In [1]:
import heapq

graph = {
    'A': [('B', 2), ('C', 4)],
    'B': [('A', 2), ('D', 5)],
    'C': [('A', 4), ('D', 1), ('E', 7)],
    'D': [('B', 5), ('C', 1), ('F', 3)],
    'E': [('C', 7), ('F', 2), ('G', 6)],
    'F': [('D', 3), ('E', 2), ('H', 4)],
    'G': [('E', 6), ('H', 3)],
    'H': [('F', 4), ('G', 3), ('T', 5)],
    'T': [('H', 5)]
}
# эвристика с рисунка
h = {'A': 8, 'B': 7, 'C': 6, 'D': 5, 'E': 4, 'F': 3, 'G': 3, 'H': 2, 'T': 0}


def astar(graph, start, goal, h):
    # как в примере, но запоминаем порядок исследованных вершин
    open_set = [(h[start], start)]
    g_score = {start: 0}
    came_from = {}
    closed = set()
    order = []
    while open_set:
        _, current = heapq.heappop(open_set)
        if current in closed:
            continue
        closed.add(current)
        order.append(current)
        if current == goal:
            path = [current]
            while current in came_from:
                current = came_from[current]
                path.append(current)
            return path[::-1], g_score[goal], order
        for neighbor, weight in graph[current]:
            tentative_g = g_score[current] + weight
            if neighbor not in g_score or tentative_g < g_score[neighbor]:
                came_from[neighbor] = current
                g_score[neighbor] = tentative_g
                f = tentative_g + h[neighbor]
                heapq.heappush(open_set, (f, neighbor))
    return None, float('inf'), order


def dijkstra_all(graph, start):
    dist = {v: float('inf') for v in graph}
    dist[start] = 0
    pq = [(0, start)]
    while pq:
        d, v = heapq.heappop(pq)
        if d > dist[v]:
            continue
        for nb, w in graph[v]:
            if d + w < dist[nb]:
                dist[nb] = d + w
                heapq.heappush(pq, (dist[nb], nb))
    return dist

In [2]:
# 1) допустимость: сравниваем h(v) с настоящим расстоянием до T
# граф неориентированный, поэтому Дейкстра от T дает d(v, T)
real = dijkstra_all(graph, 'T')
bad = []
print('Вершина   h   d(v,T)')
for v in graph:
    ok = 'ok' if h[v] <= real[v] else 'НЕ допустима'
    print('   %s     %d     %2d   %s' % (v, h[v], real[v], ok))
    if h[v] > real[v]:
        bad.append(v)
if bad:
    print('Эвристика недопустима для вершин:', bad)
else:
    print('Эвристика допустима для всех вершин')

# согласованность из подсказки: h(u) <= w(u,v) + h(v) для каждого ребра
cons = all(h[u] <= w + h[v] for u in graph for v, w in graph[u])
print('Согласованность выполняется для всех ребер:', cons)

Вершина   h   d(v,T)
   A     8     17   ok
   B     7     17   ok
   C     6     13   ok
   D     5     12   ok
   E     4     11   ok
   F     3      9   ok
   G     3      8   ok
   H     2      5   ok
   T     0      0   ok
Эвристика допустима для всех вершин
Согласованность выполняется для всех ребер: True


In [3]:
# 2) A* с заданной эвристикой
path, cost, order = astar(graph, 'A', 'T', h)
print('A* с эвристикой h:')
print('  путь:', ' -> '.join(path), ', стоимость', cost)
print('  исследованы:', ' '.join(order), '- всего', len(order))

# 3) A* при h = 0, это Дейкстра
zero = {v: 0 for v in graph}
path0, cost0, order0 = astar(graph, 'A', 'T', zero)
print('A* при h = 0 (Дейкстра):')
print('  путь:', ' -> '.join(path0), ', стоимость', cost0)
print('  исследованы:', ' '.join(order0), '- всего', len(order0))

A* с эвристикой h:
  путь: A -> C -> D -> F -> H -> T , стоимость 17
  исследованы: A B C D F E H T - всего 8
A* при h = 0 (Дейкстра):
  путь: A -> C -> D -> F -> H -> T , стоимость 17
  исследованы: A B C D F E H G T - всего 9
